In [3]:
# ================================================================
# 📊 ARABIC EYASE DATA PIPELINE — CLEAN VERSION
# ================================================================

# ================================
# 🔹 1. Mount Drive
# ================================
from google.colab import drive
drive.mount('/content/drive')

# ================================
# 🔹 2. Imports
# ================================
import os
import numpy as np
import librosa
from collections import Counter

# ================================
# 🔹 3. PATHS
# ================================
DATA_PATH = "/content/drive/MyDrive/SER_Project/"
EYASE_FOLDER = DATA_PATH + "EYASE/"

SAVE_X_PATH = DATA_PATH + "X_eyase.npy"
SAVE_Y_PATH = DATA_PATH + "y_eyase.npy"

# ================================
# 🔹 4. EMOTION LABELS (EYASE ONLY)
# ================================
EMOTION_LABELS = ["neutral", "happy", "sad", "angry"]
LABEL2IDX = {label: idx for idx, label in enumerate(EMOTION_LABELS)}

emotion_map_eyase = {
    "neu": "neutral",
    "hap": "happy",
    "sad": "sad",
    "ang": "angry"
}

# ================================
# 🔹 5. AUDIO LOADING
# ================================
def load_audio(file_path, sr=16000, max_len=5):
    audio, _ = librosa.load(file_path, sr=sr)

    max_samples = sr * max_len

    if len(audio) < max_samples:
        audio = np.pad(audio, (0, max_samples - len(audio)))
    else:
        audio = audio[:max_samples]

    return audio

# ================================
# 🔹 6. FEATURE EXTRACTION (MFCC / RAW AUDIO)
# ================================
def extract_features(file_path):
    return load_audio(file_path)

# ================================
# 🔹 7. LOAD EYASE DATASET (FIXED FOR YOUR STRUCTURE)
# ================================
def load_eyase(root_folder):
    X, y = [], []
    skipped = 0

    for speaker_folder in os.listdir(root_folder):
        speaker_path = os.path.join(root_folder, speaker_folder)

        if not os.path.isdir(speaker_path):
            continue

        for file in os.listdir(speaker_path):

            if not file.endswith(".wav"):
                continue

            try:
                # Example: fm01_ang (1).wav
                name = os.path.splitext(file)[0]

                # take emotion part (ang / sad / hap / neu)
                emotion_code = name.split("_")[1][:3].lower()

                label = emotion_map_eyase.get(emotion_code)

                if label in LABEL2IDX:
                    path = os.path.join(speaker_path, file)

                    audio = extract_features(path)

                    X.append(audio)
                    y.append(LABEL2IDX[label])
                else:
                    skipped += 1

            except Exception:
                skipped += 1

    print(f"EYASE Loaded: {len(X)} | Skipped: {skipped}")
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

# ================================
# 🔹 8. LOAD DATA
# ================================
print("⚙️ Processing EYASE dataset...")

X, y = load_eyase(EYASE_FOLDER)

# ================================
# 🔹 9. SAVE DATA
# ================================
np.save(SAVE_X_PATH, X)
np.save(SAVE_Y_PATH, y)

# ================================
# 🔹 10. VERIFICATION (SAFE VERSION)
# ================================
print("\n===== VERIFICATION =====")

print("X shape:", X.shape)
print("y shape:", y.shape)

if len(X) > 0:
    print("Value range:", X.min(), "to", X.max())

    print("\nClass distribution:")
    counts = Counter(y.tolist())

    for i, label in enumerate(EMOTION_LABELS):
        print(f"{label:<10s}: {counts.get(i,0)}")

else:
    print("⚠️ No data loaded — check folder structure or file naming!")

print("\n✅ READY FOR TRAINING")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚙️ Processing EYASE dataset...
EYASE Loaded: 579 | Skipped: 0

===== VERIFICATION =====
X shape: (579, 80000)
y shape: (579,)
Value range: -0.23305371 to 0.23617396

Class distribution:
neutral   : 150
happy     : 132
sad       : 147
angry     : 150

✅ READY FOR TRAINING
